# GemmaFit E2B Training Pipeline Preflight

Run this before spending GPU time on E2B router training. It verifies that the v5.2 dataset targets are compact JSON tool calls and that the training notebook uses completion-only / assistant-only loss instead of full-prompt loss.

In [ ]:
# Cell 0: Clone or update the repo.
import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get('GEMMAFIT_REPO_URL', 'https://github.com/bkttt0429/GemmaFit.git')
BRANCH = os.environ.get('GEMMAFIT_BRANCH', 'codex/e2b-video-capability-probes')
REPO_DIR = Path('/content/GemmaFit')

def run(cmd, cwd=None):
    print('$', ' '.join(str(part) for part in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    run(['git', 'fetch', 'origin', BRANCH])
    run(['git', 'checkout', BRANCH])
    run(['git', 'pull', '--ff-only', 'origin', BRANCH])
else:
    os.chdir('/content')
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, 'GemmaFit'])
    os.chdir(REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'rev-parse', '--short', 'HEAD'])

In [ ]:
# Cell 1: Generate or refresh the deterministic v5.2 audit dataset if needed.
import os
import subprocess
from pathlib import Path

os.chdir('/content/GemmaFit')
DATASET = Path('finetune/data/gemmafit_v5_2_e2b_evidence_router.json')

if not DATASET.exists() or DATASET.stat().st_size < 1024:
    subprocess.run([
        'python', 'finetune/data/generate_v5_e2b_evidence_router.py',
        '--out', 'finetune/data/gemmafit_v5_2_e2b_evidence_router.json',
        '--hard-cases',
        '--tool-contract-v2',
        '--zh-tw-ratio', '0.45',
        '--schema-fuzz-ratio', '0.25',
        '--validate',
    ], check=True)
else:
    print('Dataset exists:', DATASET, DATASET.stat().st_size)


In [ ]:
# Cell 2: Run fail-fast preflight. Training should not start unless this passes.
import os
import subprocess

os.chdir('/content/GemmaFit')
subprocess.run([
    'python', 'finetune/audit_e2b_training_pipeline.py',
    '--dataset', 'finetune/data/gemmafit_v5_2_e2b_evidence_router.json',
    '--notebook', 'finetune/train_e2b_v5_evidence_router.ipynb',
    '--base-model', os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it'),
    '--max-seq-length', os.environ.get('MAX_SEQ_LENGTH', '3072'),
    '--token-limit', os.environ.get('TOKEN_AUDIT_LIMIT', '2048'),
    '--strict',
    '--output', '/content/e2b_training_preflight_report.json',
], check=True)

import json
from pathlib import Path
report = json.loads(Path('/content/e2b_training_preflight_report.json').read_text())
print(json.dumps({
    'passed': report['passed'],
    'failures': report['failures'],
    'target_error_count': report['target_audit']['target_error_count'],
    'uses_full_text_field': report['notebook_audit']['uses_full_text_field'],
    'has_assistant_only_or_completion_only_config': report['notebook_audit']['has_assistant_only_or_completion_only_config'],
    'eval_max_new_tokens_default_256_detected': report['notebook_audit']['eval_max_new_tokens_default_256_detected'],
    'token_audit': report.get('token_audit'),
}, indent=2))

assert report['passed'], report['failures']


Expected pass summary:

```json
{
  "passed": true,
  "failures": [],
  "target_error_count": 0,
  "uses_full_text_field": false,
  "has_assistant_only_or_completion_only_config": true,
  "eval_max_new_tokens_default_256_detected": false
}
```

If this passes, continue with `finetune/train_e2b_v5_evidence_router.ipynb` and run the training phase. If it fails, do not train.